# CatBoost Regression

training catboost regression in blocks using baseline features + traficom features


## 1. Imports

In [50]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load data

In [51]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [52]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

for dataset_df in [train_df, validation_df]:
    for date_column in ["first_seen_date", "last_seen_date", "scrape_date"]:
        if date_column in dataset_df.columns:
            dataset_df[date_column] = pd.to_datetime(dataset_df[date_column], errors="coerce")

reference_first_seen_date = min(
    df["first_seen_date"].min()
    for df in [train_df, validation_df]
    if "first_seen_date" in df.columns
)

for dataset_df in [train_df, validation_df]:
    dataset_df["first_seen_day_offset"] = (
        dataset_df["first_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["last_seen_day_offset"] = (
        dataset_df["last_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["listing_midpoint_day_offset"] = (
        dataset_df["first_seen_day_offset"]
        + dataset_df["last_seen_day_offset"]
    ) / 2


## 3. Quick data checks

In [53]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 82)
Validation shape: (1689, 82)


In [54]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 82 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   product_id                      7954 non-null   int64         
 1   part_name                       7954 non-null   object        
 2   price                           7954 non-null   float64       
 3   quality_grade                   7954 non-null   object        
 4   oem_number                      7954 non-null   object        
 5   mileage                         7954 non-null   float64       
 6   brand                           7954 non-null   object        
 7   model                           7954 non-null   object        
 8   category                        7954 non-null   object        
 9   subcategory                     7954 non-null   object        
 10  scrape_date                     7954 non-null   datetime64[ns

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [55]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column in train_df.columns
]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
]

catboost_leakage_risk_feature_reasons = {
    "times_observed": "Uses the full listing observation count, which is only known after the listing lifecycle has unfolded.",
    "observed_span_days": "Uses the full observed listing span, which reaches into the future for earlier snapshots.",
    "price_changed_flag": "Summarizes whether the listing price ever changed across the whole listing history.",
    "price_change_count": "Counts price changes over the full listing history, including future changes.",
    "absolute_price_change": "Uses the total absolute price movement across the complete listing history.",
    "relative_price_change_pct": "Uses the net percentage price movement across the complete listing history.",
    "price_changed_flag_so_far": "Uses the current row's price to decide whether the listing has changed price so far, so it leaks target information.",
    "price_change_count_so_far": "Counts price changes including the current row's price, so it leaks target information.",
    "absolute_price_change_so_far": "Computed from the current row's price minus the first observed price, so it directly encodes the target.",
    "relative_price_change_pct_so_far": "Computed from the current row's price relative to the first observed price, so it directly encodes the target.",
    "last_seen_day_offset": "Numeric version of the listing end date and therefore future-looking.",
    "listing_midpoint_day_offset": "Derived from both listing start and end, so it still leaks future listing duration.",
}
catboost_leakage_risk_features = set(catboost_leakage_risk_feature_reasons)
catboost_all_date_offset_features = {
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
}

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_day_offset": "Leakage-safe numeric listing-start offset retained for trusted CatBoost comparisons.",
    "last_seen_day_offset": "Numeric position of listing end within the crawl window; retained only for exploratory comparisons.",
    "listing_midpoint_day_offset": "Numeric midpoint of listing visibility window; retained only for exploratory comparisons.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

catboost_specific_exclusion_reasons = {
    "first_seen_date": "Raw date field replaced with numeric day-offset features for CatBoost.",
    "last_seen_date": "Raw date field replaced with numeric day-offset features for CatBoost.",
}

recommended_excluded_features = set(recommended_exclusion_reasons)
catboost_excluded_features = recommended_excluded_features | set(catboost_specific_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in catboost_excluded_features
]

recommended_model_features_without_date_offsets = [
    column for column in recommended_model_features
    if column not in catboost_all_date_offset_features
]

manual_all_feature_groups_leakage_safe = [
    column for column in manual_all_feature_groups
    if column not in catboost_leakage_risk_features
]

recommended_model_features_leakage_safe = [
    column for column in recommended_model_features
    if column not in catboost_leakage_risk_features
]

recommended_model_features_leakage_safe_without_date_offsets = [
    column for column in recommended_model_features_leakage_safe
    if column not in catboost_all_date_offset_features
]

recommended_model_features_leakage_safe_with_first_seen_offset = list(dict.fromkeys(
    recommended_model_features_leakage_safe_without_date_offsets
    + ["first_seen_day_offset"]
))

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
    + [
        {
            "feature": column,
            "status": "catboost_specific_exclusion",
            "reason": catboost_specific_exclusion_reasons[column],
        }
        for column in sorted(catboost_specific_exclusion_reasons)
    ]
    + [
        {
            "feature": column,
            "status": "trusted_selection_exclusion",
            "reason": catboost_leakage_risk_feature_reasons[column],
        }
        for column in sorted(catboost_leakage_risk_features)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nCatBoost-specific exclusions:")
print(sorted(catboost_specific_exclusion_reasons))

print("\nLeakage-risk features excluded from trusted CatBoost selection:")
print(sorted(catboost_leakage_risk_features))

print("\nNumber of recommended model features:", len(recommended_model_features))
print("Number of trusted recommended model features:", len(recommended_model_features_leakage_safe_with_first_seen_offset))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
3,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
4,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
5,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
6,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
7,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...
8,first_seen_date,catboost_specific_exclusion,Raw date field replaced with numeric day-offse...
9,last_seen_date,catboost_specific_exclusion,Raw date field replaced with numeric day-offse...


## 5. Select baseline feature set

In [56]:
features_baseline_only = list(dict.fromkeys(
    [
        feature for feature in baseline_features
        if feature not in catboost_excluded_features
    ]
    + [
        "first_seen_day_offset",
    ]
))

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 14


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset']

## 6. Build X and y

In [57]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [58]:
y_train_log = np.log(y_train)
y_validation_log = np.log(y_validation)

y_train_log.head()


0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [59]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'first_seen_day_offset']

## 8. Detect column types

In [60]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [61]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,6,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,8,"[part_name, quality_grade, oem_number, brand, ..."


## 9. Tune Native CatBoost


In [62]:
catboost_candidate_params = {
    "raw_rmse_reference": {
        "target_mode": "raw",
        "model_params": {
            "loss_function": "RMSE",
            "eval_metric": "MAE",
            "iterations": 2200,
            "learning_rate": 0.04,
            "depth": 6,
            "l2_leaf_reg": 8,
            "random_strength": 1.0,
            "bagging_temperature": 0.5,
            "border_count": 254,
            "one_hot_max_size": 10,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Best trusted direction in the focused benchmark: raw target with plain RMSE trees.",
    },
    "raw_rmse_depth7": {
        "target_mode": "raw",
        "model_params": {
            "loss_function": "RMSE",
            "eval_metric": "MAE",
            "iterations": 2000,
            "learning_rate": 0.035,
            "depth": 7,
            "l2_leaf_reg": 10,
            "random_strength": 1.0,
            "bagging_temperature": 0.75,
            "border_count": 254,
            "one_hot_max_size": 10,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Slightly deeper raw-target RMSE variant for richer interactions.",
    },
    "raw_mae_reference": {
        "target_mode": "raw",
        "model_params": {
            "loss_function": "MAE",
            "eval_metric": "MAE",
            "iterations": 1800,
            "learning_rate": 0.04,
            "depth": 6,
            "l2_leaf_reg": 8,
            "random_strength": 1.0,
            "bagging_temperature": 0.5,
            "border_count": 254,
            "one_hot_max_size": 10,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Raw-target MAE optimization as a direct MAE-focused baseline.",
    },
    "log_rmse_reference": {
        "target_mode": "log",
        "model_params": {
            "loss_function": "RMSE",
            "eval_metric": "MAE",
            "iterations": 2200,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 8,
            "random_strength": 1.0,
            "bagging_temperature": 0.5,
            "border_count": 254,
            "one_hot_max_size": 10,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Log-target control to confirm whether CatBoost prefers direct-price fitting.",
    },
    "raw_mae_depth7_plain": {
        "target_mode": "raw",
        "model_params": {
            "loss_function": "MAE",
            "eval_metric": "MAE",
            "iterations": 2200,
            "learning_rate": 0.030,
            "depth": 7,
            "l2_leaf_reg": 6,
            "random_strength": 0.5,
            "bagging_temperature": 0.5,
            "border_count": 254,
            "one_hot_max_size": 12,
            "rsm": 0.85,
            "boosting_type": "Plain",
        },
        "notes": "Requested raw-target MAE check with deeper trees and broader one-hot handling.",
    },
    "raw_rmse_wider_onehot": {
        "target_mode": "raw",
        "model_params": {
            "loss_function": "RMSE",
            "eval_metric": "MAE",
            "iterations": 2400,
            "learning_rate": 0.025,
            "depth": 6,
            "l2_leaf_reg": 7,
            "random_strength": 0.8,
            "bagging_temperature": 0.6,
            "border_count": 254,
            "one_hot_max_size": 15,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Requested raw-target RMSE check with wider one-hot categorical handling.",
    },
    "log_rmse_depth6_plain": {
        "target_mode": "log",
        "model_params": {
            "loss_function": "RMSE",
            "eval_metric": "MAE",
            "iterations": 2200,
            "learning_rate": 0.030,
            "depth": 6,
            "l2_leaf_reg": 8,
            "random_strength": 1.0,
            "bagging_temperature": 0.5,
            "border_count": 254,
            "one_hot_max_size": 12,
            "rsm": 0.8,
            "boosting_type": "Plain",
        },
        "notes": "Requested log-target RMSE check with the broader one-hot categorical window.",
    },
}


def prepare_catboost_frame(df):
    prepared_df = df.copy()

    datetime_columns = prepared_df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()
    if len(datetime_columns) > 0:
        prepared_df = prepared_df.drop(columns=datetime_columns)

    object_columns = prepared_df.select_dtypes(include=["object", "category"]).columns.tolist()
    for column in object_columns:
        prepared_df[column] = prepared_df[column].astype("string")

    bool_columns = prepared_df.select_dtypes(include=["bool"]).columns.tolist()
    for column in bool_columns:
        prepared_df[column] = prepared_df[column].astype(int)

    return prepared_df


def get_catboost_cat_features(prepared_df):
    return prepared_df.select_dtypes(include=["string"]).columns.tolist()


def make_catboost_regressor(params=None):
    model_params = {
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": -1,
    }
    if params is not None:
        model_params.update(params)

    return CatBoostRegressor(**model_params)


def prepare_catboost_target(y_series, target_mode):
    if target_mode == "log":
        return np.log(y_series)
    if target_mode == "raw":
        return y_series.copy()

    raise ValueError(f"Unsupported target_mode: {target_mode}")


def convert_catboost_predictions_to_eur(predictions, target_mode):
    predictions = np.asarray(predictions, dtype=float)

    if target_mode == "log":
        clipped_predictions = np.nan_to_num(
            predictions,
            nan=0.0,
            posinf=np.log(1_000_000.0),
            neginf=np.log(1e-3),
        )
        clipped_predictions = np.clip(
            clipped_predictions,
            a_min=np.log(1e-3),
            a_max=np.log(1_000_000.0),
        )
        return np.exp(clipped_predictions)

    if target_mode == "raw":
        return np.clip(
            np.nan_to_num(predictions, nan=0.0, posinf=0.0, neginf=0.0),
            a_min=0.0,
            a_max=None,
        )

    raise ValueError(f"Unsupported target_mode: {target_mode}")


def fit_catboost_model(X_train_current, y_train_current, X_validation_current, y_validation_current, params):
    X_train_prepared_current = prepare_catboost_frame(X_train_current)
    X_validation_prepared_current = prepare_catboost_frame(X_validation_current)
    cat_features_current = get_catboost_cat_features(X_train_prepared_current)

    model_current = make_catboost_regressor(params)
    model_current.fit(
        X_train_prepared_current,
        y_train_current,
        cat_features=cat_features_current,
        eval_set=(X_validation_prepared_current, y_validation_current),
        use_best_model=True,
        early_stopping_rounds=120,
    )

    return (
        model_current,
        X_train_prepared_current,
        X_validation_prepared_current,
        cat_features_current,
    )


In [63]:
catboost_tuning_feature_sets = {
    "trusted_baseline_registry_stack": {
        "features": list(dict.fromkeys(
            features_baseline_only
            + traficom_features
            + registry_lifecycle_features
        )),
        "trusted_for_selection": True,
        "priority": 1,
    },
    "trusted_recommended_no_offsets": {
        "features": recommended_model_features_leakage_safe_without_date_offsets,
        "trusted_for_selection": True,
        "priority": 2,
    },
    "trusted_baseline_registry_stack_without_oem_number": {
        "features": [
            column for column in list(dict.fromkeys(
                features_baseline_only
                + traficom_features
                + registry_lifecycle_features
            ))
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
        "priority": 3,
    },
    "trusted_recommended_no_offsets_without_oem_number": {
        "features": [
            column for column in recommended_model_features_leakage_safe_without_date_offsets
            if column != "oem_number"
        ],
        "trusted_for_selection": True,
        "priority": 4,
    },
}

tuning_results = []
for feature_variant_name, feature_variant_config in catboost_tuning_feature_sets.items():
    tuning_features = feature_variant_config["features"]
    X_train_tuning = train_df[tuning_features].copy()
    X_validation_tuning = validation_df[tuning_features].copy()

    for config_name, config in catboost_candidate_params.items():
        y_train_tuning = prepare_catboost_target(y_train, config["target_mode"])
        y_validation_tuning = prepare_catboost_target(y_validation, config["target_mode"])
        (
            model_current,
            X_train_tuning_prepared,
            X_validation_tuning_prepared,
            cat_features_tuning,
        ) = fit_catboost_model(
            X_train_tuning,
            y_train_tuning,
            X_validation_tuning,
            y_validation_tuning,
            config["model_params"],
        )

        validation_pred_model_scale_current = model_current.predict(X_validation_tuning_prepared)
        validation_pred_current = convert_catboost_predictions_to_eur(
            validation_pred_model_scale_current,
            config["target_mode"],
        )

        tuning_results.append({
            "config": config_name,
            "feature_variant": feature_variant_name,
            "feature_variant_priority": feature_variant_config["priority"],
            "trusted_for_selection": feature_variant_config["trusted_for_selection"],
            "target_mode": config["target_mode"],
            "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
            "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
            "validation_R2": r2_score(y_validation, validation_pred_current),
            "best_iteration": model_current.get_best_iteration(),
            "notes": config["notes"],
        })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_MAE", "validation_RMSE"],
    ascending=[True, True],
).reset_index(drop=True)

selected_catboost_config_name = tuning_results_df.iloc[0]["config"]
selected_catboost_feature_variant_name = tuning_results_df.iloc[0]["feature_variant"]
selected_catboost_config = catboost_candidate_params[selected_catboost_config_name]
selected_catboost_target_mode = selected_catboost_config["target_mode"]
selected_catboost_params = selected_catboost_config["model_params"]

print("Selected CatBoost config:", selected_catboost_config_name)
print("Selected CatBoost tuning feature variant:", selected_catboost_feature_variant_name)
print(selected_catboost_config)

display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)


Selected CatBoost config: raw_rmse_depth7
Selected CatBoost tuning feature variant: trusted_recommended_no_offsets
{'target_mode': 'raw', 'model_params': {'loss_function': 'RMSE', 'eval_metric': 'MAE', 'iterations': 2000, 'learning_rate': 0.035, 'depth': 7, 'l2_leaf_reg': 10, 'random_strength': 1.0, 'bagging_temperature': 0.75, 'border_count': 254, 'one_hot_max_size': 10, 'rsm': 0.8, 'boosting_type': 'Plain'}, 'notes': 'Slightly deeper raw-target RMSE variant for richer interactions.'}


,config,feature_variant,feature_variant_priority,trusted_for_selection,target_mode,validation_MAE,validation_RMSE,validation_R2,best_iteration,notes
0,raw_rmse_depth7,trusted_recommended_no_offsets,2,True,raw,44.6120,90.8654,0.9743,1999,Slightly deeper raw-target RMSE variant for richer interactions.
1,raw_rmse_reference,trusted_recommended_no_offsets,2,True,raw,49.9948,106.9905,0.9644,2198,Best trusted direction in the focused benchmark: raw target with plain RMSE trees.
2,raw_rmse_depth7,trusted_baseline_registry_stack,1,True,raw,53.8260,123.4317,0.9526,1951,Slightly deeper raw-target RMSE variant for richer interactions.
3,raw_rmse_reference,trusted_baseline_registry_stack,1,True,raw,55.3890,130.1154,0.9473,2199,Best trusted direction in the focused benchmark: raw target with plain RMSE trees.
4,log_rmse_reference,trusted_recommended_no_offsets,2,True,log,61.4087,195.8291,0.8807,2199,Log-target control to confirm whether CatBoost prefers direct-price fitting.
5,log_rmse_reference,trusted_baseline_registry_stack,1,True,log,66.4967,220.3894,0.8489,2193,Log-target control to confirm whether CatBoost prefers direct-price fitting.
6,raw_mae_reference,trusted_baseline_registry_stack,1,True,raw,83.5141,274.5152,0.7656,1119,Raw-target MAE optimization as a direct MAE-focused baseline.
7,raw_mae_reference,trusted_recommended_no_offsets,2,True,raw,96.7219,350.1223,0.6187,897,Raw-target MAE optimization as a direct MAE-focused baseline.


## 10. Train tuned baseline CatBoost


In [ ]:
baseline_model, X_train_prepared, X_validation_prepared, baseline_cat_features = fit_catboost_model(
    X_train,
    prepare_catboost_target(y_train, selected_catboost_target_mode),
    X_validation,
    prepare_catboost_target(y_validation, selected_catboost_target_mode),
    selected_catboost_params,
)

## 11. Predict and convert back to euro scale

In [ ]:
validation_pred_model_scale = baseline_model.predict(X_validation_prepared)

In [ ]:
validation_pred = convert_catboost_predictions_to_eur(
    validation_pred_model_scale,
    selected_catboost_target_mode,
)

## 12. Preview Baseline CatBoost Predictions


In [ ]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,299.441733
1,473.6,286.813786
2,142.1,147.717912
3,82.9,61.184081
4,177.6,118.829282


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [ ]:
features_baseline_plus_traficom = list(dict.fromkeys(
    features_baseline_only + traficom_features
))

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 27


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [ ]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [ ]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Prepare and train baseline + Traficom CatBoost


In [ ]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()


In [ ]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

(
    catboost_traficom,
    X_train_traficom_prepared,
    X_validation_traficom_prepared,
    cat_features_traficom,
) = fit_catboost_model(
    X_train_traficom,
    prepare_catboost_target(y_train, selected_catboost_target_mode),
    X_validation_traficom,
    prepare_catboost_target(y_validation, selected_catboost_target_mode),
    selected_catboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 17. Predict baseline + Traficom CatBoost


In [ ]:
validation_pred_model_scale_traficom = catboost_traficom.predict(X_validation_traficom_prepared)

## 18. Predict on validation and convert back to euro scale

In [ ]:
validation_pred_traficom = convert_catboost_predictions_to_eur(
    validation_pred_model_scale_traficom,
    selected_catboost_target_mode,
)

In [ ]:
validation_pred_traficom = convert_catboost_predictions_to_eur(
    validation_pred_model_scale_traficom,
    selected_catboost_target_mode,
)

## 19. Preview Baseline + Traficom CatBoost Predictions


In [ ]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,281.216830
1,473.6,310.637279
2,142.1,157.769835
3,82.9,59.064015
4,177.6,116.143501


In [ ]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,251.130784
std,567.153248,536.718231
min,5.900000,0.000000
25%,59.000000,51.980760
50%,100.600000,89.192869
75%,236.000000,246.863428
max,4284.000000,4385.409622


## 20. All Recommended Features

This section trains the CatBoost model on every recommended feature while keeping the same validation structure used across the other notebooks.


In [ ]:
features_all = recommended_model_features_leakage_safe_with_first_seen_offset

print("Number of trusted recommended model features:", len(features_all))
features_all

Number of trusted recommended model features: 65


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [ ]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [ ]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Prepare and train recommended-feature CatBoost


In [ ]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()


In [ ]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

(
    catboost_all,
    X_train_all_prepared,
    X_validation_all_prepared,
    cat_features_all,
) = fit_catboost_model(
    X_train_all,
    prepare_catboost_target(y_train, selected_catboost_target_mode),
    X_validation_all,
    prepare_catboost_target(y_validation, selected_catboost_target_mode),
    selected_catboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 24. Predict recommended-feature CatBoost


In [ ]:
validation_pred_model_scale_all = catboost_all.predict(X_validation_all_prepared)

## 25. Predict on validation and convert back to euro scale

In [ ]:
validation_pred_all = convert_catboost_predictions_to_eur(
    validation_pred_model_scale_all,
    selected_catboost_target_mode,
)

In [ ]:
validation_pred_all = convert_catboost_predictions_to_eur(
    validation_pred_model_scale_all,
    selected_catboost_target_mode,
)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [ ]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,271.831795
1,473.6,319.154417
2,142.1,155.909408
3,82.9,55.931733
4,177.6,114.002374


In [ ]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,252.513264
std,567.153248,537.780999
min,5.900000,0.000000
25%,59.000000,49.114287
50%,100.600000,94.842752
75%,236.000000,244.512652
max,4284.000000,4345.871252


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing CatBoost model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [ ]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",271.831795,94.231795,8879.631125
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",319.154417,154.445583,23853.438121
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",155.909408,13.809408,190.699740
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",55.931733,26.968267,727.287449
4,177.6,airbag,all,"Side airbags - , e-(Right)",114.002374,63.597626,4044.658070


In [ ]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [ ]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,19.76,16.39,24.49,27.1%,57.5%,93.4%
1,electric / transmitter / databox / sensor,404,28.91,18.01,51.72,36.6%,55.2%,82.4%
2,vehicle exterior / suspension,294,40.02,13.23,91.07,39.8%,68.0%,86.4%
3,brakes,214,47.31,20.25,87.69,21.0%,49.5%,77.1%
4,airbag,106,48.53,29.45,75.27,7.5%,31.1%,71.7%
5,gear box / drive axle / middle axle,241,98.64,34.36,194.51,12.0%,38.6%,55.2%
6,engine,249,102.53,25.92,233.18,30.5%,42.6%,61.0%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,102.53,25.92,233.18,30.5%,42.6%,61.0%
5,gear box / drive axle / middle axle,241,98.64,34.36,194.51,12.0%,38.6%,55.2%
4,airbag,106,48.53,29.45,75.27,7.5%,31.1%,71.7%
3,brakes,214,47.31,20.25,87.69,21.0%,49.5%,77.1%
2,vehicle exterior / suspension,294,40.02,13.23,91.07,39.8%,68.0%,86.4%
1,electric / transmitter / databox / sensor,404,28.91,18.01,51.72,36.6%,55.2%,82.4%
0,fuel,181,19.76,16.39,24.49,27.1%,57.5%,93.4%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,255.53,177.97,330.99,0.0%,4.8%,4.8%
22,passenger airbag,25,91.37,29.52,134.36,4.0%,12.0%,64.0%
21,left rear,50,62.02,27.27,109.62,14.0%,42.0%,66.0%
20,right rear,49,58.78,17.49,95.86,14.3%,63.3%,75.5%
19,contact roll airbag,20,55.25,55.30,62.62,0.0%,20.0%,35.0%
18,adaptiv farthållare sensor,24,54.10,54.31,54.15,0.0%,0.0%,4.2%
17,gear box bracket,31,44.14,50.49,47.04,0.0%,19.4%,41.9%
16,right front,48,35.67,22.27,47.79,22.9%,39.6%,68.8%
15,left front,39,33.73,34.36,43.84,15.4%,46.2%,79.5%
14,sensor ac inner temperature,24,23.13,9.27,30.60,54.2%,54.2%,75.0%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,8.38,7.54,9.15,60.0%,100.0%,100.0%
1,Trailing link rear -(Left),20,10.44,5.10,14.32,70.0%,70.0%,100.0%
2,ABS Hydraulic pump -,24,13.42,11.77,16.67,50.0%,62.5%,100.0%
3,Wheel bearing spindle shaft -(Right rear),20,17.59,10.21,24.87,50.0%,75.0%,100.0%
4,Wheel bearing spindle shaft -(Left rear),21,17.69,12.97,22.15,38.1%,71.4%,100.0%
5,Drive shaft -(Left front),32,21.83,17.15,25.87,31.2%,56.2%,96.9%
6,Brake Caliper -(Left front),20,42.83,39.01,47.53,0.0%,5.0%,70.0%
7,Drive shaft -(Right front),23,50.75,62.16,60.84,21.7%,30.4%,43.5%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Drive shaft -(Right front),23,50.75,62.16,60.84,21.7%,30.4%,43.5%
6,Brake Caliper -(Left front),20,42.83,39.01,47.53,0.0%,5.0%,70.0%
5,Drive shaft -(Left front),32,21.83,17.15,25.87,31.2%,56.2%,96.9%
4,Wheel bearing spindle shaft -(Left rear),21,17.69,12.97,22.15,38.1%,71.4%,100.0%
3,Wheel bearing spindle shaft -(Right rear),20,17.59,10.21,24.87,50.0%,75.0%,100.0%
2,ABS Hydraulic pump -,24,13.42,11.77,16.67,50.0%,62.5%,100.0%
1,Trailing link rear -(Left),20,10.44,5.10,14.32,70.0%,70.0%,100.0%
0,Distributors Vacuum regulator -,20,8.38,7.54,9.15,60.0%,100.0%,100.0%


## 29. Select The Best CatBoost Feature Set

Reuse the tuned CatBoost hyperparameters across every feature-set experiment so the comparison stays focused on feature value instead of model-setup noise.


In [ ]:
excluded_features = catboost_excluded_features

feature_sets = {
    "baseline only": {
        "features": features_baseline_only,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core": {
        "features": features_baseline_plus_traficom,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle": {
        "features": features_baseline_plus_traficom + registry_lifecycle_features,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle + traficom_extended": {
        "features": (
            features_baseline_plus_traficom
            + registry_lifecycle_features
            + traficom_extended_features
        ),
        "trusted_for_selection": True,
    },
    "trusted manual all-features set": {
        "features": manual_all_feature_groups_leakage_safe,
        "trusted_for_selection": True,
    },
    "trusted recommended features": {
        "features": recommended_model_features_leakage_safe_with_first_seen_offset,
        "trusted_for_selection": True,
    },
    "trusted recommended features without date offsets": {
        "features": recommended_model_features_leakage_safe_without_date_offsets,
        "trusted_for_selection": True,
    },
    "exploratory previous manual all-features set": {
        "features": manual_all_feature_groups,
        "trusted_for_selection": False,
    },
    "exploratory all recommended features": {
        "features": recommended_model_features,
        "trusted_for_selection": False,
    },
}

# Reset derived state so rerunning this section does not reuse stale results.
experiment_results = []
experiment_feature_details = []
experiment_predictions = {}
for stale_name in [
    "best_experiment_name",
    "best_model_validation_df",
    "best_model_direction_summary",
    "best_model_price_band_summary",
    "experiment_results_df",
    "trusted_experiment_results_df",
    "best_validation_predictions",
]:
    if stale_name in globals():
        del globals()[stale_name]

In [ ]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


In [ ]:
for model_name, feature_config in feature_sets.items():
    feature_list = feature_config["features"]
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    (
        model_current,
        X_train_current_prepared,
        X_validation_current_prepared,
        cat_features_current,
    ) = fit_catboost_model(
        X_train_current,
        prepare_catboost_target(y_train, selected_catboost_target_mode),
        X_validation_current,
        prepare_catboost_target(y_validation, selected_catboost_target_mode),
        selected_catboost_params,
    )

    validation_pred_model_scale_current = model_current.predict(X_validation_current_prepared)
    validation_pred_current = convert_catboost_predictions_to_eur(
        validation_pred_model_scale_current,
        selected_catboost_target_mode,
    )
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "trusted_for_selection": feature_config["trusted_for_selection"],
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "trusted_for_selection": feature_config["trusted_for_selection"],
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 30. Validate The Best CatBoost Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [ ]:
# Rebuild the best-model validation dataframe from the current experiment results.
if "experiment_results" not in globals() or len(experiment_results) == 0:
    raise NameError(
        "experiment_results is not defined. Run the feature-set comparison section first."
    )

experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,229,33.0,53.7,20.73
1,"(5.899, 47.4]",underpredicted,120,40.3,32.1,8.22
2,"(47.4, 82.6]",overpredicted,122,59.3,70.9,11.53
3,"(47.4, 82.6]",underpredicted,213,62.2,47.4,14.74
4,"(82.6, 154.06]",overpredicted,110,122.8,164.6,41.77
5,"(82.6, 154.06]",underpredicted,219,103.4,77.9,25.50
6,"(154.06, 248.42]",overpredicted,224,201.7,289.9,88.21
7,"(154.06, 248.42]",underpredicted,114,211.9,162.7,49.15
8,"(248.42, 4284.0]",overpredicted,104,677.3,745.1,67.81
9,"(248.42, 4284.0]",underpredicted,234,975.7,879.6,96.05


In [ ]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
if "experiment_results" not in globals() or len(experiment_results) == 0:
    raise NameError(
        "experiment_results is not defined. Run the feature-set comparison section first."
    )

experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,16.43,23.29,66.8%,-7.09
1,"(47.4, 82.6]",335,47.6,82.6,61.1,13.57,18.72,22.3%,+3.45
2,"(82.6, 154.06]",329,82.8,153.9,109.9,30.94,49.92,27.9%,+11.97
3,"(154.06, 248.42]",338,154.1,248.3,205.1,75.04,124.68,38.4%,-19.19
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,87.36,149.60,12.4%,+22.91


In [ ]:
experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

baseline_candidates_df = trusted_experiment_results_df.loc[
    trusted_experiment_results_df["experiment"] == "baseline only"
]
if baseline_candidates_df.empty:
    baseline_candidates_df = experiment_results_df.loc[
        experiment_results_df["experiment"] == "baseline only"
    ]
if baseline_candidates_df.empty:
    raise NameError("Could not find the 'baseline only' row in experiment_results.")

baseline_row = baseline_candidates_df.iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]

display_columns = [
    "experiment",
    "trusted_for_selection",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

experiment_results_df.sort_values(
    ["trusted_for_selection", "validation_MAE", "validation_RMSE"],
    ascending=[False, True, True],
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})

,experiment,trusted_for_selection,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
6,trusted recommended features without date offsets,True,64,44.6120,-6.9837,3,90.8654,-28.1280,3,0.9743
1,baseline + traficom_core,True,27,51.2503,-0.3454,4,118.5196,-0.4738,4,0.9563
0,baseline only,True,14,51.5957,+0.0000,5,118.9935,+0.0000,5,0.9560
3,baseline + traficom_core + registry_lifecycle + traficom_extended,True,63,53.3023,+1.7066,6,127.6072,+8.6137,7,0.9493
2,baseline + traficom_core + registry_lifecycle,True,41,53.8260,+2.2303,7,123.4317,+4.4382,6,0.9526
5,trusted recommended features,True,65,54.2263,+2.6306,8,129.9677,+10.9742,8,0.9475
4,trusted manual all-features set,True,63,54.3064,+2.7107,9,134.1664,+15.1730,9,0.9440
7,exploratory previous manual all-features set,False,71,21.2351,-30.3606,1,88.7619,-30.2316,2,0.9755
8,exploratory all recommended features,False,73,21.8225,-29.7732,2,87.5285,-31.4649,1,0.9762
